In [3]:
import PyPDF2

from langchain_core.prompts import ChatPromptTemplate
from langchain_community.llms.ollama import Ollama

from IPython.display import display, Markdown, clear_output
import time

In [4]:
#initialize the llama model via ollama
model_id = "llama3.1"
model = Ollama(model=model_id)

C:\Users\USER\AppData\Local\Temp\ipykernel_24536\3073641258.py:3: LangChainDeprecationWarning: The class `Ollama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaLLM``.
  model = Ollama(model=model_id)


In [5]:
# function to extract text from PDF
def extract_textPfrom_pdf(pdf_path):
    with open(pdf_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        text = ""
        for page in range(len(reader.pages)):
            text += reader.pages[page].extract_text()
    return text

In [6]:
def generate_refined_summary(model, text, max_iterations=5):
    iteration =0
    current_summary = None
    questions_generated = True
    
    # generate an initial summary
    initial_prompt_text = """
    You are a summarization expert. Your task is to write summaries.
    Here is the original document text:
    
    {text}
    
    Start by creating a summary in your first pass.
    
    Create an initial summary.
    """
    initial_prompt = ChatPromptTemplate.from_template(initial_prompt_text)
    current_summary = model(initial_prompt.format(text=text))
    
    display(Markdown(f"### Initial Summary (Iteration {iteration})"))
    display(Markdown(current_summary))
    
    #iterative refinment process
    while iteration < max_iterations and questions_generated:
        print("=======================Next iteration=======================")
        iteration += 1
        # ask the LLM to compare the original text and summary and refine it
        refinement_prompt_text = """
        You are a summarizationn expert. Your task is to refine summaries.
        Here is the original document text:
        
        {text}
        
        Refine the below current summary, keep it as it is but ensure it becomes more complete, coherent, clear and accurate.
        Aim to capture the essence of the text with each refinement,
        
        Current summary:
        
        {summary}
        
        Please provide a refined summary below:
        """
        refinement_prompt = ChatPromptTemplate.from_template(refinement_prompt_text)
        current_summary = model(refinement_prompt.format(text=text, summary = current_summary))
    
    return current_summary        
    